In [1]:
import pandas as pd
from rdkit import Chem
from rdkit.Chem import Descriptors

df = pd.read_csv("/Users/marczaiser/Desktop/Studium /Masterstudium/2. Semester/DSA103_Advanced Chemical Data Science/ChemicalDataScience103_Fork_Zaiser_Marc/assignments/session12_assignment/Solution/Escerichia Coli.csv", sep=";")

# 2) Select only the needed columns
df = df[
    [
        "Molecule ChEMBL ID",
        "Smiles",
        "Standard Type",
        "Standard Relation",
        "Standard Value",
        "Standard Units",
        "Assay ChEMBL ID",
    ]
].copy()

# 3) Rename columns to cleaner names
df.columns = [
    "Molecule_chembl_id",
    "Smiles",
    "Standard_Type",
    "Standard_relation",
    "Standard_value",
    "Standard_units",
    "Assay_chembl_id",
]

# 4) Drop rows with no SMILES
df = df.dropna(subset=["Smiles"]).copy()
df["smiles"] = df["Smiles"].str.strip()
df = df[df["Smiles"] != ""]

# Rows without SMILES string are being removed as they cannot be used for downstream analysis (ie mol. weight calculation)

# 5) Filter by molecular weight range
def calculate_mw(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    return Descriptors.MolWt(mol)

df["mw"] = df["smiles"].apply(calculate_mw)

# Example range: molecular weight between 200 and 400
min_mw = 200
max_mw = 400
df = df[(df["mw"] >= min_mw) & (df["mw"] <= max_mw)].copy()

# 6) Randomly sample a few hundred entries
n_sample = min(300, len(df))
df_sample = df.sample(n=n_sample, random_state=42)


# 7) Export sample DataFrame to CSV
output_file = r"/Users/marczaiser/Desktop/Studium /Masterstudium/2. Semester/DSA103_Advanced Chemical Data Science/ChemicalDataScience103_Fork_Zaiser_Marc/assignments/session12_assignment/Solution/chembl_sample_300_mw_200_400.csv" 

df_sample.to_csv(output_file, index=False)

print(f"Saved {len(df_sample)} entries to: {output_file}")

/var/folders/9t/prw_ys6j3tn4w8gd17bhy2tr0000gn/T/ipykernel_41515/268224747.py:5: DtypeWarning: Columns (29: Assay Cell Type) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("/Users/marczaiser/Desktop/Studium /Masterstudium/2. Semester/DSA103_Advanced Chemical Data Science/ChemicalDataScience103_Fork_Zaiser_Marc/assignments/session12_assignment/Solution/Escerichia Coli.csv", sep=";")
[14:05:28] Explicit valence for atom # 1 P, 7, is greater than permitted


Saved 300 entries to: /Users/marczaiser/Desktop/Studium /Masterstudium/2. Semester/DSA103_Advanced Chemical Data Science/ChemicalDataScience103_Fork_Zaiser_Marc/assignments/session12_assignment/Solution/chembl_sample_300_mw_200_400.csv


In [3]:
import pubchempy as pcp
import pandas as pd
from tqdm import tqdm
import time
from pathlib import Path


# define ROOT directory as parent folder
ROOT = Path(__file__).resolve().parents[0]
DATA = ROOT/"Solution"

# Load your ChEMBL dataset
chembl_df = pd.read_csv(DATA / "/Users/marczaiser/Desktop/Studium /Masterstudium/2. Semester/DSA103_Advanced Chemical Data Science/ChemicalDataScience103_Fork_Zaiser_Marc/assignments/session12_assignment/Solution/Escerichia Coli.csv")  # use your actual filename and path
smiles_list = chembl_df["smiles"].dropna().tolist() # use the column tag with the "smiles"

records = []

for smi in tqdm(smiles_list):  # limit to 200 compounds for testing, tqdm just adds progress bar
    try:
        compounds = pcp.get_compounds(smi, namespace='smiles') # look for each smiles-code in the list sliced from the ChEMBL data
        if not compounds:
            continue
        
        c = compounds[0]
        records.append({
            "smiles": smi, # make sure that the column name for the smiles code matches the chembl dataframe!
            "CID": c.cid,
            "MolecularWeight": c.molecular_weight,
            "XLogP": c.xlogp,
            "TPSA": c.tpsa,
            "HBondDonorCount": c.h_bond_donor_count,
            "HBondAcceptorCount": c.h_bond_acceptor_count,
            "RotatableBondCount": c.rotatable_bond_count,
            "HeavyAtomCount": c.heavy_atom_count
        })
        
        # delay to avoid flooding PubChem servers
        time.sleep(0.2)
        
    except Exception as e:
        print(f"Failed for {smi[:20]}...: {e}")

pubchem_df = pd.DataFrame(records)
pubchem_df.to_csv(r"/Users/marczaiser/Desktop/Studium /Masterstudium/2. Semester/DSA103_Advanced Chemical Data Science/ChemicalDataScience103_Fork_Zaiser_Marc/assignments/session12_assignment/Solution/Escerichia Coli.csv", index=False) # use your path and specified filename
print(f"Saved {len(pubchem_df)} entries.")


#------------------------------------------------------------

NameError: name '__file__' is not defined